In [0]:
print(
    "Watermark Gold Records :",
    spark.table("watermark_gold_revenue").count()
)

Watermark Gold Records : 58


In [0]:
display(
    spark.table("watermark_gold_revenue")
)

window,daily_revenue
"List(2024-02-05T00:00:00.000Z, 2024-02-06T00:00:00.000Z)",98618.0
"List(2024-01-13T00:00:00.000Z, 2024-01-14T00:00:00.000Z)",114433.0
"List(2024-02-04T00:00:00.000Z, 2024-02-05T00:00:00.000Z)",85656.0
"List(2024-01-31T00:00:00.000Z, 2024-02-01T00:00:00.000Z)",77823.0
"List(2024-01-14T00:00:00.000Z, 2024-01-15T00:00:00.000Z)",79664.0
"List(2024-01-01T00:00:00.000Z, 2024-01-02T00:00:00.000Z)",56799.0
"List(2024-01-09T00:00:00.000Z, 2024-01-10T00:00:00.000Z)",88910.0
"List(2024-01-12T00:00:00.000Z, 2024-01-13T00:00:00.000Z)",116176.0
"List(2024-01-30T00:00:00.000Z, 2024-01-31T00:00:00.000Z)",90242.0
"List(2024-01-16T00:00:00.000Z, 2024-01-17T00:00:00.000Z)",87487.0


In [0]:
(
    watermark_gold_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/late_transaction_data/checkpoint/watermark"
    )
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("watermark_gold_revenue")
)

In [0]:
print(watermark_gold_df.isStreaming)

True


In [0]:
watermark_gold_df = (
    watermarked_df
    .groupBy(
        window("txn_date", "1 day")
    )
    .agg(
        sum("amount").alias("daily_revenue")
    )
)

In [0]:
watermarked_df = (
    watermark_df
    .withWatermark(
        "txn_date",
        "1 day"
    )
)

In [0]:
watermark_df.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- txn_date: timestamp (nullable = true)
 |-- amount: string (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
watermark_df = (
    watermark_df
    .withColumn(
        "txn_date",
        to_timestamp("txn_date")
    )
)

In [0]:
print(watermark_df.isStreaming)

True


In [0]:
watermark_df = (
    spark.readStream
    .format("delta")
    .table("bronze_transactions")
)

In [0]:
from pyspark.sql.functions import *